In [ ]:
# ─── 1. CÀI ĐẶT MÔI TRƯỜNG ───────────────────────────────────────────────────
!pip uninstall -y peft accelerate transformers -q
!pip install -q \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.16.1 \
  safetensors==0.4.2 \
  scikit-learn \
  Pillow \
  numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 9.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not cur

In [ ]:

# ─── 2. SETUP & IMPORTS ───────────────────────────────────────────────────────
import os, time, functools, json
import torch
import numpy as np
from PIL import Image
from torch import nn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import (CLIPProcessor, CLIPModel, TrainingArguments,
                          Trainer, EarlyStoppingCallback)
from transformers.modeling_outputs import SequenceClassifierOutput
from datasets import load_dataset, Image as HFImage
from google.colab import drive

# Tối ưu bộ nhớ GPU
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Kết nối Drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/CLIP_ViT_B32_FFT_IFND_Final"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Save to:", SAVE_DIR)

# ─── 3. DATASET & PREPROCESS ──────────────────────────────────────────────────
print("⏳ Loading dataset IFND-multimodal...")
hf_dataset = load_dataset("Nhat243/IFND-multimodal")
hf_dataset = hf_dataset.cast_column("image", HFImage(decode=True))

model_name = "openai/clip-vit-base-patch32"
processor  = CLIPProcessor.from_pretrained(model_name)

def preprocess(example):
    try:
        image = example['image'].convert("RGB")
    except:
        image = Image.new("RGB", (224, 224), (128, 128, 128))

    inputs = processor(
        text=str(example['text']),
        images=image,
        truncation=True,
        padding="max_length",
        max_length=77,
        return_tensors="pt"
    )
    return {
        "input_ids":      inputs["input_ids"][0],
        "attention_mask": inputs["attention_mask"][0],
        "pixel_values":   inputs["pixel_values"][0],
        "labels":         int(example["label"])
    }

print("⏳ Preprocessing dataset...")
encoded_dataset = hf_dataset.map(
    preprocess,
    remove_columns=hf_dataset["train"].column_names,
    desc="Preprocessing"
)
encoded_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "pixel_values", "labels"])

train_dataset = encoded_dataset["train"]
val_dataset   = encoded_dataset["validation"]
test_dataset  = encoded_dataset["test"]
print(f"✅ Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# ─── 4. MODEL ARCHITECTURE (FULL FINE-TUNING + GATED FUSION + ALIGNMENT) ──────
class CLIPForMFND(nn.Module):
    def __init__(self, model_name, num_labels=2, lambda_weight=0.1, delta=0.5):
        super().__init__()
        # Load mô hình gốc (Mặc định FFT sẽ mở khóa toàn bộ tham số)
        self.clip = CLIPModel.from_pretrained(model_name)
        embed_dim = self.clip.config.projection_dim # 512

        # Gated Fusion & Alignment Head
        self.cross_attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=8, batch_first=True)
        self.W_g = nn.Linear(embed_dim * 2, embed_dim)
        self.classifier = nn.Linear(embed_dim, num_labels)

        self.lambda_weight = lambda_weight
        self.delta = delta
        self.align_loss_fn = nn.CosineEmbeddingLoss(margin=delta)

    def forward(self, input_ids=None, attention_mask=None, pixel_values=None, labels=None):
        outputs = self.clip(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            return_dict=True
        )
        h_T, h_I = outputs.text_embeds, outputs.image_embeds

        # --- GATED MODALITY FUSION ---
        attn_out, _ = self.cross_attention(h_T.unsqueeze(1), h_I.unsqueeze(1), h_I.unsqueeze(1))
        z = attn_out.squeeze(1)

        gate = torch.sigmoid(self.W_g(torch.cat([h_T, h_I], dim=1)))
        h_f = gate * z

        # --- CLASSIFICATION & LOSS ---
        logits = self.classifier(h_f)
        loss = None
        if labels is not None:
            loss_cls = nn.CrossEntropyLoss()(logits, labels)

            # Giữ nguyên Align Loss theo các bản trước để đảm bảo tính đối chứng
            loss_align = self.align_loss_fn(h_T, h_I, labels.float() * 2 - 1)

            loss = loss_cls + (self.lambda_weight * loss_align)

        return SequenceClassifierOutput(loss=loss, logits=logits)

import gc; gc.collect(); torch.cuda.empty_cache()
model = CLIPForMFND(model_name).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Total params    : {total_params:,}")
print(f"📊 Trainable params: {trainable_params:,} (FFT Mode)")

# ─── 5. METRICS & TRAINING ────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds  = np.argmax(probs, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": p,
        "recall":    r,
        "f1":        f1,
        "auc":       roc_auc_score(labels, probs[:, 1])
    }

torch.load = functools.partial(torch.load, weights_only=False)

training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=1e-5, # FFT dùng LR thấp hơn PEFT
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("\n🚀 Bắt đầu huấn luyện FFT...")
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
start_train = time.time()
trainer.train()

trainer.save_model(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
train_time_min = (time.time() - start_train) / 60

# ─── 6. INFERENCE MEASUREMENT (BATCH SIZE = 1, P50 MEDIAN) ────────────────────
model.eval()
dummy_image = Image.new("RGB", (224, 224), (128, 128, 128))
dummy_text  = "Sample news for latency measurement."
inputs = processor(text=dummy_text, images=dummy_image, truncation=True, padding="max_length", max_length=77, return_tensors="pt")
input_ids = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)
pixel_values = inputs["pixel_values"].to(device)

if device == "cuda": torch.cuda.synchronize()
with torch.no_grad():
    for _ in range(50): model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)

latencies = []
with torch.no_grad():
    for _ in range(200):
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)
        if device == "cuda": torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
vram = torch.cuda.max_memory_allocated() / 1024**3 if device == "cuda" else 0

# ─── 7. FINAL EVALUATION & SAVE RESULTS ───────────────────────────────────────
print("\n📊 Evaluating on IFND TEST split...")
test_results = trainer.predict(test_dataset)
logits = test_results.predictions
labels = test_results.label_ids
probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
preds  = np.argmax(probs, axis=1)

p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
acc = accuracy_score(labels, preds)
auc = roc_auc_score(labels, probs[:, 1])

final_results = {
    "Model": "CLIP ViT-B/32",
    "Method": "Full Fine-Tuning (FFT)",
    "Dataset": "IFND-multimodal",
    "Hardware_Stats": {
        "Total_Params_M": round(total_params / 1e6, 2),
        "Training_Time_Min": round(train_time_min, 2),
        "Peak_VRAM_GB": round(vram, 2),
        "Latency_P50_ms": round(np.median(latencies), 2)
    },
    "Test_Metrics": {
        "Accuracy": round(acc * 100, 2),
        "Precision": round(p * 100, 2),
        "Recall": round(r * 100, 2),
        "F1_Score": round(f1 * 100, 2),
        "AUC": round(auc, 4)
    }
}

json_path = os.path.join(SAVE_DIR, "results_FFT_IFND.json")
with open(json_path, "w") as f:
    json.dump(final_results, f, indent=4)

print(f"\n✅ Đã lưu kết quả tại: {json_path}")
print(f"F1 Score: {final_results['Test_Metrics']['F1_Score']}% | AUC: {auc:.4f}")
print(f"{'='*50}")

# ─── 8. AUTO-DISCONNECT GPU ───────────────────────────────────────────────────
print("⏳ Đang ngắt kết nối GPU...")
from google.colab import runtime
time.sleep(5)
runtime.unassign()

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Using device: cuda
Mounted at /content/drive
Save to: /content/drive/MyDrive/CLIP_ViT_B32_FFT_IFND_Final
⏳ Loading dataset IFND-multimodal...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/8416 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1052 [00:00<?, ? examples/s]

...

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
1,0.575000,0.480843,0.797529,0.790972,1.000000,0.883288,0.985798
2,0.461500,0.448247,0.797529,0.790972,1.000000,0.883288,0.918777
3,0.404400,0.452606,0.780418,0.837838,0.884615,0.860591,0.840674


Checkpoint destination directory /content/drive/MyDrive/CLIP_ViT_B32_FFT_IFND_Final/checkpoint-526 already exists and is non-empty.Saving will proceed but saved results may be invalid.



📊 Evaluating on IFND TEST split...



✅ Đã lưu kết quả tại: /content/drive/MyDrive/CLIP_ViT_B32_FFT_IFND_Final/results_FFT_IFND.json
F1 Score: 88.44% | AUC: 0.9840
⏳ Đang ngắt kết nối GPU...
